# 11. Live traffic-sign recognition from a USB webcam

Use the existing **CNV-W1A1 road-sign model** on the **PYNQ FPGA**, with a USB camera such as the Logitech C270. No training, Vivado build, or package reinstallation is needed.

The supplied classifier has **43 traffic-sign categories plus one “Not a roadsign” output**. It classifies the selected image region; it does not automatically find traffic signs in a street scene. The additional background class can also make mistakes.

By the end, you will compare recognition from an image file and a camera, observe the effect of framing, distance, lighting, and tilt, and distinguish FPGA inference timing from application FPS.

Based on Xilinx BNN-PYNQ: https://github.com/Xilinx/BNN-PYNQ . Original files and notices remain under the repository LICENSE.

## 1. Prepare the board and import the packages

1. Run notebook **08_CNV-BNN_Road-Signs.ipynb** successfully first.
2. Shut down the kernels of other FPGA/camera notebooks. Connect the C270 to the **PYNQ board**.
3. Have a printed sign or a sign displayed on a **phone** ready. STOP is a useful first example. Keep the PC screen for the notebook.
4. Run the next cell with **Shift + Enter**. It imports the packages; it does not start the camera.

In [ ]:
import contextlib
import io
import time
import textwrap
import cv2
import numpy as np
from PIL import Image as PILImage
from IPython.display import display, Image as NotebookImage
import bnn

# BNN-PYNQ uses Image.ANTIALIAS, renamed in newer Pillow releases.
if not hasattr(PILImage, "ANTIALIAS"):
    PILImage.ANTIALIAS = PILImage.Resampling.LANCZOS

print("OpenCV:", cv2.__version__)
print("BNN module:", bnn.__file__)

## 2. Set the experiment parameters

**What this cell does:** stores the camera, model, and session settings used later. It does not open the camera or run inference.

**What you need to do:** leave the defaults unchanged for the first test and run the next cell with **Shift + Enter**.

The capture requests 640 × 480 pixels. The session will stop at **30 seconds or 300 frames**, whichever comes first. `ROI_FRACTION` sets the size of the central input region; its default value is 0.65. `CENTER_CROP = False` can be tested later to use the whole frame.

In [ ]:
CAMERA_DEVICE = "/dev/video0"
CAPTURE_WIDTH = 640
CAPTURE_HEIGHT = 480
WARMUP_FRAMES = 5
MAX_SECONDS = 30.0
MAX_FRAMES = 300
CENTER_CROP = True
ROI_FRACTION = 0.65  # Fraction of the shorter camera dimension.
JPEG_QUALITY = 75
NETWORK = bnn.NETWORK_CNVW1A1
RUNTIME = bnn.RUNTIME_HW
DATASET = "road-signs"

assert MAX_SECONDS > 0 and MAX_FRAMES > 0
assert 0 < ROI_FRACTION <= 1
assert 1 <= JPEG_QUALITY <= 100

## 3. Load the road-sign model on the FPGA

Run the next cell with **Shift + Enter** and wait for the “Ready” message. It loads the existing `road-signs` parameters with CNV-W1A1. The camera is not started here.

Keep the same classifier throughout this notebook. Do not load a different FPGA overlay in another kernel while this notebook is running.

In [ ]:
if DATASET not in bnn.available_params(NETWORK):
    raise RuntimeError("Road-sign parameters are missing for " + NETWORK)
classifier = bnn.CnvClassifier(NETWORK, DATASET, RUNTIME)
print("Ready:", NETWORK, "| dataset:", DATASET, "| FPGA runtime")
print("Outputs:", len(classifier.classes),
      "(repository model: 43 signs plus 'Not a roadsign')")
print("The camera has not started yet.")

## 4. Check recognition on two image files

Run the next cell. It displays the existing STOP and end-of-no-overtaking images with the expected label and the FPGA prediction. **Green** means that the class matches; **red** means it differs.

This confirms that the model can be called before adding camera conditions. Missing reference images are downloaded from this repository once. If these file-based predictions fail, resolve that first before judging camera results.

For camera practice you may print these images, or open them on a phone:

- STOP: https://github.com/onigas/BNN-PYNQ/blob/master/notebooks/pictures/road_signs/stop.jpg
- End of no overtaking: https://github.com/onigas/BNN-PYNQ/blob/master/notebooks/pictures/road_signs/end_no_overtaking.png

In [ ]:
from pathlib import Path
import urllib.request
import matplotlib.pyplot as plt

picture_dir = Path("/home/xilinx/jupyter_notebooks/bnn/pictures/road_signs")
references = [
    ("stop.jpg", 14),
    ("end_no_overtaking.png", 41),
]
fig, axes = plt.subplots(1, len(references), figsize=(8, 3.5))
for ax, (filename, expected_id) in zip(axes, references):
    picture = picture_dir / filename
    if not picture.exists():
        picture.parent.mkdir(parents=True, exist_ok=True)
        url = "https://raw.githubusercontent.com/onigas/BNN-PYNQ/master/notebooks/pictures/road_signs/" + filename
        urllib.request.urlretrieve(url, str(picture))
    with PILImage.open(str(picture)) as opened:
        image = opened.convert("RGB")
    with contextlib.redirect_stdout(io.StringIO()):
        predicted_id = int(classifier.classify_image(image.copy()))
    ax.imshow(image)
    ax.axis("off")
    expected = classifier.class_name(expected_id)
    predicted = classifier.class_name(predicted_id)
    ax.set_title("Expected: " + expected, fontsize=10)
    ax.text(0.5, -0.12, "FPGA: " + predicted, transform=ax.transAxes,
            ha="center", fontsize=10,
            color="green" if predicted_id == expected_id else "red",
            wrap=True)
fig.suptitle("File-based check before using the camera")
fig.tight_layout(rect=(0, 0.05, 1, 0.94))
plt.show()

## 5. Define the camera and display functions

**What this cell does:** defines functions for capture, central cropping, BGR → RGB conversion, resizing to 32 × 32, FPGA classification, and display.

**What you need to do:** run the next cell with **Shift + Enter**, without editing it. No camera picture is expected yet.

The display reports:
- **Accelerator ms:** inference timing reported by the BNN runtime.
- **Call ms:** wall time of the Python classification call, including formatting and temporary-file handling.
- **Application FPS:** completed iterations per elapsed second, including capture, preparation, inference, annotation, JPEG encoding, and sending notebook updates. Browser rendering and network delivery are not measured.

FPS shown during the loop describes preceding completed iterations. The summary includes the last completed update. Class scores are not calibrated probabilities, so no confidence percentage is displayed.

The camera is released on normal completion, Python exceptions, or a handled interrupt. A blocked native driver call can delay interruption or the time limit.

In [ ]:
def read_frame(cap):
    ok, frame = cap.read()
    if not ok or frame is None or frame.size == 0:
        raise RuntimeError(
            "No camera frame. Check the USB connection and shut down "
            "other camera notebook kernels."
        )
    return frame


def prepare_input(frame, center_crop):
    height, width = frame.shape[:2]
    if center_crop:
        side = max(1, int(min(height, width) * ROI_FRACTION))
        x0, y0 = (width - side) // 2, (height - side) // 2
        region = frame[y0:y0 + side, x0:x0 + side]
        box = (x0, y0, side)
    else:
        region, box = frame, None
    rgb = cv2.cvtColor(region, cv2.COLOR_BGR2RGB)
    rgb32 = cv2.resize(rgb, (32, 32), interpolation=cv2.INTER_AREA)
    return PILImage.fromarray(rgb32), rgb32, box


def run_live(classifier):
    cap = None
    completed = 0
    accelerator_ms, call_ms = [], []
    start = None
    elapsed = 0.0
    interrupted = False
    try:
        cap = cv2.VideoCapture(CAMERA_DEVICE, cv2.CAP_V4L2)
        if not cap.isOpened():
            raise RuntimeError("Cannot open " + str(CAMERA_DEVICE)
                               + ". Check the camera and other kernels.")
        cap.set(cv2.CAP_PROP_FRAME_WIDTH, CAPTURE_WIDTH)
        cap.set(cv2.CAP_PROP_FRAME_HEIGHT, CAPTURE_HEIGHT)
        # Best effort: not all V4L2 drivers support this setting.
        cap.set(cv2.CAP_PROP_BUFFERSIZE, 1)
        for _ in range(WARMUP_FRAMES):
            read_frame(cap)
        print("Camera opened. Stop early with Kernel > Interrupt.")
        view = display(NotebookImage(data=cv2.imencode(
            ".jpg", np.zeros((48, 320, 3), dtype=np.uint8)
        )[1].tobytes()), display_id=True)
        start = time.perf_counter()
        while completed < MAX_FRAMES and time.perf_counter() - start < MAX_SECONDS:
            frame = read_frame(cap)
            image, rgb32, box = prepare_input(frame, CENTER_CROP)
            call_start = time.perf_counter()
            # Suppress the library's Python timing prints on each frame.
            with contextlib.redirect_stdout(io.StringIO()):
                class_id = int(classifier.classify_image(image))
            wall_ms = (time.perf_counter() - call_start) * 1000.0
            hardware_ms = float(classifier.usecPerImage) / 1000.0
            label = classifier.class_name(class_id)
            fps = completed / elapsed if elapsed > 0 else 0.0

            annotated = frame.copy()
            if box is not None:
                x, y, side = box
                cv2.rectangle(annotated, (x, y), (x + side - 1, y + side - 1),
                              (0, 255, 0), 2)
            # A fixed panel keeps text readable at different capture sizes.
            annotated = cv2.resize(annotated, (480, 360))
            panel = np.zeros((160, 480, 3), dtype=np.uint8)
            label_lines = textwrap.wrap(label, width=56)[:3]
            lines = (
                ["Predicted class: {}".format(class_id)]
                + label_lines
                + ["Accelerator: {:.2f} ms | Call: {:.2f} ms".format(
                    hardware_ms, wall_ms),
                   "Application: {:.2f} FPS | Frames: {}".format(fps, completed)]
            )
            for i, text in enumerate(lines):
                cv2.putText(panel, text, (8, 22 + i * 23),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.43, (255, 255, 255), 1,
                            cv2.LINE_AA)
            inset = cv2.resize(cv2.cvtColor(rgb32, cv2.COLOR_RGB2BGR),
                               (96, 96), interpolation=cv2.INTER_NEAREST)
            annotated[254:350, 374:470] = inset
            cv2.putText(annotated, "32x32 input", (370, 247),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.43, (0, 255, 255), 1)
            canvas = np.vstack((panel, annotated))
            ok, jpeg = cv2.imencode(".jpg", canvas,
                                    [cv2.IMWRITE_JPEG_QUALITY, JPEG_QUALITY])
            if not ok:
                raise RuntimeError("JPEG encoding failed.")
            view.update(NotebookImage(data=jpeg.tobytes()))
            completed += 1
            accelerator_ms.append(hardware_ms)
            call_ms.append(wall_ms)
            elapsed = time.perf_counter() - start
    except KeyboardInterrupt:
        interrupted = True
        print("Session interrupted.")
    finally:
        if cap is not None:
            cap.release()
        print("Camera released.")

    result = {
        "frames": completed,
        "elapsed_s": elapsed,
        "application_fps": completed / elapsed if elapsed > 0 else 0.0,
        "mean_accelerator_ms": float(np.mean(accelerator_ms)) if completed else None,
        "mean_classify_call_ms": float(np.mean(call_ms)) if completed else None,
        "interrupted": interrupted,
    }
    print("Completed frames: {} | measured loop time: {:.2f} s | {:.2f} FPS".format(
        completed, elapsed, result["application_fps"]))
    if completed:
        print("Mean accelerator: {:.2f} ms | mean classify call: {:.2f} ms".format(
            result["mean_accelerator_ms"], result["mean_classify_call_ms"]))
    return result

## 6. Start the camera and classify signs

**Run the next cell with Shift + Enter.** The camera image appears below it.

1. A **green square** marks the input region. Hold the printed sign or phone so that the complete sign is inside this square, roughly filling it with a small margin.
2. Keep the sign upright and use a simple background. Avoid phone-screen reflections. Start with STOP.
3. Read the predicted class and name above the camera image. The small inset shows the exact **32 × 32** input, enlarged for inspection.
4. Replace the sign and observe how the prediction changes. The green square remains fixed; it is not an object-detection box.
5. The session stops after **30 seconds or 300 processed frames**. To stop sooner, use **Kernel → Interrupt**.
6. Run **only this session cell** again to repeat the experiment.

The busy marker (`[*]`) is expected while frames update. After stopping, the camera is released and timing statistics are printed. A slow read or inference can delay completion. If an interrupt cannot stop a blocked native call, restart the kernel and shut down other camera kernels before retrying.

If the display says **Not a roadsign**, adjust the framing and lighting; this is a model prediction, not proof that no sign is present. Other sign labels can also be predicted on an empty background.

In [ ]:
session_stats = run_live(classifier)

## 7. Student experiments

**A. Different signs.** Test at least three signs. Start with the two reference images; add a speed-limit sign printed or displayed on a phone. Record the true sign and the observed prediction.

**B. Distance and framing.** Use the same sign close to the camera and farther away. Inspect the 32 × 32 inset. Compare `CENTER_CROP = True` and `False`: rerun the settings cell after changing a value, then rerun only the session cell. You do not need to reload the model.

**C. Tilt and illumination.** Tilt a sign gently, then reduce illumination or add a reflection. Find a condition where the label becomes unstable. Do not assume that a successful upright example will work at every angle.

| True sign | Framing / condition | Observed prediction | Stable or fluctuating? |
|---|---|---|---|
| STOP | Upright, close | | |
| STOP | Farther away | | |
| STOP | Tilted | | |
| End of no overtaking | Upright, close | | |

**Timing question:** why can `1000 / accelerator_ms` be much higher than the measured application FPS? Use the three displayed measurements in your explanation.

These camera observations are not an accuracy benchmark. For a measured accuracy percentage, labelled test samples and a defined evaluation procedure are needed.

## Troubleshooting

- **No camera frame / cannot open device:** check the board's USB connection, shut down other camera kernels, and verify `/dev/video0`.
- **Unstable labels:** center the complete sign, make it larger, use a plain background, and reduce glare. Look at the input inset.
- **Long labels:** names are wrapped across lines in the display.
- **File recognition works but camera recognition fails:** compare camera framing, lighting, reflections, and the much smaller input representation.
- **Model or overlay error:** verify notebook 08 first. This lab reuses the same installed model and hardware.
- **Cell remains busy without updates:** interrupt; if blocked in native code, restart the kernel before another attempt.

**Validation status:** notebook code, cropping, rendering paths, and camera cleanup are checked with simulated devices. Actual C270/FPGA predictions and performance require testing on the board.